# Model Baseline Development (Decision Tree)

## 1. Setup

In [1]:
import pandas as pd
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score,
    classification_report
)

## 2. Data Loading

In [2]:
input_path = Path("../data/processed/dataset_final.csv")
df = pd.read_csv(input_path)
print("Data loaded. Shape:", df.shape)
display(df.head())

Data loaded. Shape: (1200, 41)


,transaction_type_online purchase,transaction_type_transfer,transaction_type_withdrawal,merchant_category_Clothing,merchant_category_Crypto,merchant_category_Electronics,merchant_category_Entertainment,merchant_category_Food,merchant_category_Gambling,merchant_category_Services,...,account_age_days,description_risk_score,transaction_frequency_24h,num_failed_transactions_7d,transaction_hour,is_weekend,num_devices_used,num_countries_used,kyc_verified,is_fraud
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,-0.929790,-0.368841,0.0,0.0,8.0,0.0,2.0,0.0,0.0,0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.761706,-0.368841,0.0,0.0,7.0,0.0,2.0,0.0,1.0,0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.964482,-0.368841,0.0,0.0,19.0,0.0,1.0,0.0,0.0,1
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.560900,3.310755,0.0,0.0,3.0,0.0,1.0,1.0,1.0,1
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.064750,-0.368841,0.0,0.0,10.0,0.0,2.0,0.0,0.0,0


## 3. Data Preparation

In [3]:
# The dataset_final.csv is already largely preprocessed.
# Below are typical data preparation steps that would be applied
# if starting from raw data, but are largely for documentation here
# as the input CSV is already in a suitable format.

# 1. Feature Selection / Dropping Irrelevant Columns
# These columns are typically dropped if present, but may already be absent
# in the preprocessed 'dataset_final.csv'.
cols_to_drop = [
    'transaction_id', 'timestamp', 'customer_id',
    'transaction_description', 'merchant_name', 'customer_support_note',
    'risk_category'
 ]
df_clean = df.drop(columns=cols_to_drop, errors='ignore')

# 2. Encoding Categorical Variables
# Assuming one-hot encoding has already been applied upstream to create dataset_final.csv.
# If not, a step like the following would be needed:
# df_encoded = pd.get_dummies(df_clean, drop_first=True)
# For this notebook, df_clean is already df_encoded as per dataset_final.csv structure.
X_features = df_clean.copy() # Use df_clean as the basis for features

In [4]:
# 3. Split X (Features) and y (Target)
X = X_features.drop('is_fraud', axis=1) # Drop the target from the features
y = df['is_fraud']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (960, 40)
Testing set shape: (240, 40)


## 4. Model Training

In [6]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


## 5. Model Evaluation

In [8]:
y_pred = dt_model.predict(X_test)
y_prob = dt_model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))


Accuracy: 0.7792
ROC-AUC: 0.6851

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.85      0.86       188
           1       0.49      0.52      0.50        52

    accuracy                           0.78       240
   macro avg       0.68      0.69      0.68       240
weighted avg       0.78      0.78      0.78       240



## 6. Save Results

In [9]:
model_name = "Decision Tree"
output_file = "../results/model_baseline_results.csv"

# Calculate metrics
metrics = {
    'model': model_name,
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob)
}

# Create DataFrame
results_df = pd.DataFrame([metrics])

# Ensure directory exists
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Save to CSV
results_df.to_csv(output_file, index=False)

print(f"Results for {model_name} saved to {output_file}")

Results for Decision Tree saved to ../results/model_baseline_results.csv


## 7. Business Insights

*(Placeholder for business insights based on the Decision Tree model's performance and coefficients.)*